In [3]:
import pyomo.environ as pyo

# Create a model
model = ConcreteModel()

# Sets
model.designer = Set(initialize=["designer_1", "designer_2", "designer_3"])
model.project   = Set(initialize=["project_1", "project_2", "project_3", "project_4"])

# Data
scoring_data = {
    ("designer_1", "project_1"): 90,
    ("designer_1", "project_2"): 80,
    ("designer_1", "project_3"): 10,
    ("designer_1", "project_4"): 50,
    ("designer_2", "project_1"): 60,
    ("designer_2", "project_2"): 70,
    ("designer_2", "project_3"): 50,
    ("designer_2", "project_4"): 65,
    ("designer_3", "project_1"): 70,
    ("designer_3", "project_2"): 40,
    ("designer_3", "project_3"): 80,
    ("designer_3", "project_4"): 85,
}

project_requirements = {
    "project_1": 70,
    "project_2": 50,
    "project_3": 85,
    "project_4": 35,
}

availability = 80

model.x = Var(model.designer, model.project, domain=pyo.NonNegativeReals)

def objective_rule(model):
    return sum(scoring_data[d,p] * model.x[d,p] for d in model.designer for p in model.project)
model.objective = Objective(rule=objective_rule, sense=maximize)

def supply_rule(model, d):
    return sum(model.x[d,p] for p in model.project) <= availability
model.supply = Constraint(model.designer, rule=supply_rule)

def demand_rule(model, p):
    return sum(model.x[d,p] for d in model.designer) >= project_requirements[p]
model.demand = Constraint(model.project, rule=demand_rule)

solver = pyo.SolverFactory('glpk')
results = solver.solve(model)

# model.display()

# Print the results
for d in model.designer:
    for p in model.project:
        val = pyo.value(model.x[d, p])
        if model.x[d,p].value > 0:
            print(f"Designer {d} assigned to Project {p} with score { val}")

NameError: name 'ConcreteModel' is not defined

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import shutil
import sys
import os.path
from pyomo.environ import *

import pyomo.environ as pe
import pyomo.opt as po

In [ ]:
# define sets and data
nodes = set(range(1, 7))

connections = {
    (1,2):1,
    (1,4):1,
    (2,1):1,
    (2,3):1,
    (2,5):1,
    (3,2):1,
    (3,5):1,
    (3,6):1,
    (4,1):1,
    (4,5):1,
    (5,2):1,
    (5,3):1,
    (5,4):1,
    (5,6):1,
    (6,3):1,
    (6,5):1,
} 

cost = {1:40,2:65,3:43,4:48,5:72,6:36}

In [ ]:
model = pe.ConcreteModel()

model.nodes = pe.Set(initialize=nodes)
adj = list(connections.keys())
model.A = pe.Set(within=model.nodes*model.nodes, initialize=adj)

model.connections = pe.Param(model.A, initialize=connections)
model.cost = pe.Param(model.nodes, initialize=cost)

model.X = pe.Var(model.nodes, domain=Binary)

obj_expr = sum(model.cost[i]*model.X[i] for i in model.nodes)
model.obj = pe.Objective(sense=pe.minimize, expr=obj_expr)

def allnodescover_rule(m,i,j):
    return (m.X[i] + m.X[j]) >= m.connections[i,j]

model.nodes_constraint = pe.Constraint(model.A, rule=allnodescover_rule)

In [ ]:
solver = po.SolverFactory('gurobi')
results = solver.solve(model, tee=True)

In [ ]:
print("Minimum Total Cost:", pe.value(model.obj))

for n in model.nodes:
    if pe.value(model.X[n]) == 1:
        print("Selected Nodes are:", n)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import shutil
import sys
import os.path
from pyomo.environ import *

import pyomo.environ as pe
import pyomo.opt as po

In [ ]:
# define sets and data
nodes = set(range(1, 7))

connections = {
    (1,2):1,
    (1,4):1,
    (2,1):1,
    (2,3):1,
    (2,5):1,
    (3,2):1,
    (3,5):1,
    (3,6):1,
    (4,1):1,
    (4,5):1,
    (5,2):1,
    (5,3):1,
    (5,4):1,
    (5,6):1,
    (6,3):1,
    (6,5):1,
} 

In [ ]:
from pyomo.common.enums import maximize
model = pe.ConcreteModel()

model.nodes = pe.Set(initialize=nodes)
adj = list(connections.keys())
model.A = pe.Set(within=model.nodes*model.nodes, initialize=adj)

model.connections = pe.Param(model.A, initialize=connections)

model.X = pe.Var(model.nodes, domain=pe.Binary)
model.Y = pe.Var(model.A, domain=pe.Binary)

def covered_links(m):
    return sum(m.Y[i,j] for (i,j) in m.A)/2
model.obj = pe.Objective(sense=pe.maximize,rule=covered_links)

def each_edge_rule(m, i, j):
    return m.Y[i,j] <= m.X[i] + m.X[j]
model.edge_constraint = Constraint(model.A, rule=each_edge_rule)

def each_node_rule_a(m, i, j):
    return m.Y[i,j] >= m.X[i]
model.node_constraint_a = Constraint(model.A, rule=each_node_rule_a)

def each_node_rule_b(m, i, j):
    return m.Y[i,j] >= m.X[j]
model.node_constraint_b = Constraint(model.A, rule=each_node_rule_b)

def maxstation_rule(m):
    return sum(m.X[i] for i in m.nodes) <= 2

model.nodes_constraint =  pe.Constraint(rule=maxstation_rule)

source (type: set).  This WILL potentially lead to nondeterministic behavior
in Pyomo


In [ ]:
solver = po.SolverFactory('gurobi')
results = solver.solve(model, tee=True)

Read LP format model from file C:\Users\ADMINI~1\AppData\Local\Temp\tmpvnhog6mw.pyomo.lp
Reading time = 0.00 seconds
x1: 49 rows, 22 columns, 118 nonzeros
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12700, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 49 rows, 22 columns and 118 nonzeros (Max)
Model fingerprint: 0xacdf386b
Model has 16 linear objective coefficients
Variable types: 0 continuous, 22 integer (22 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [5e-01, 5e-01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+00, 2e+00]
Found heuristic solution: objective -0.0000000
Presolve removed 40 rows and 8 columns
Presolve time: 0.00s
Presolved: 9 rows, 14 columns, 30 nonzeros
Variable types: 0 continuous, 14 integer (14 binary)

Root relaxation: objective 6.000000e+00, 10 it

In [ ]:
#print("Objective Value (no physical sense): ",pe.value(m.obj))
print("Number of edges covered ",pe.value(model.obj))

Number of edges covered  6.0
